In [ ]:
"""
Machine Learning Model Benchmarking for Microstructure-Based Hardness Prediction.

This script evaluates multiple regression models using K-fold cross-validation and 
compares predictive performance based on R², RMSE, and MAE metrics. 
Additionally, it performs Wilcoxon signed-rank tests to statistically validate 
the superiority of the best-performing model.
Prediction results and publication-quality figures are automatically exported.
"""

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

fontsize = 35

# =============================================================================
# 1. Initialize Output Directory & Load Cleaned Dataset
# =============================================================================

output_dir = r"./data/figure"
os.makedirs(output_dir, exist_ok=True)

df_filtered = pd.read_csv(r"./data/d_hv_with_corr_features.csv")

target_column = "HV"
drop_cols = ["FILE_NAME", target_column]
feature_cols = [c for c in df_filtered.columns if c not in drop_cols]

# Extract feature and target matrices
X = df_filtered[feature_cols].values
y = df_filtered[target_column].values

# =============================================================================
# 2. Define Regression Models & Cross-Validation
# =============================================================================

# Scale-sensitive models are wrapped in a Pipeline so scaling is fit only on
# the training fold, avoiding data leakage from cross_val_predict
models = {
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.1))]),
    "SVR": Pipeline([("scaler", StandardScaler()), ("model", SVR(kernel="rbf", C=10))]),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# =============================================================================
# 3. Model Evaluation and Visualization Pipeline
# =============================================================================

results = []
model_errors = {}  # Dictionary to store absolute errors of each model for statistical testing

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(22, 14))
axes = axes.flatten()

print(f"[INFO] Initiating model benchmarking for {len(models)} regression models...\n")
print(f"{'Model':<20} | {'R²':>8} | {'RMSE':>8} | {'MAE':>8}")
print("-" * 55)

for i, (name, model) in enumerate(models.items()):

    # Cross-validation prediction profile
    y_pred = cross_val_predict(model, X, y, cv=kf)

    # Calculate and store absolute errors for statistical testing
    abs_error = np.abs(y - y_pred)
    model_errors[name] = abs_error

    # Compute regression validation metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)

    results.append({"Model": name, "R2": r2, "RMSE": rmse, "MAE": mae})
    print(f"{name:<20} | {r2:>8.3f} | {rmse:>8.3f} | {mae:>8.3f}")

    # Determine aligned data range for identical scaling
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())

    fig_single, ax_single = plt.subplots(figsize=(10, 10))

    ax_single.scatter(y, y_pred, alpha=0.6, edgecolors="w", color="#2b5c8f", s=50)
    ax_single.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label=f"$R^2 = {r2:.3f}$")

    ax_single.set_xlabel("Actual HV", fontsize=fontsize)
    ax_single.set_ylabel("Predicted HV", fontsize=fontsize)
    ax_single.tick_params(axis="both", labelsize=fontsize)
    ax_single.legend(loc="upper left", fontsize=fontsize, frameon=True)
    ax_single.grid(True, linestyle=":", alpha=0.6)

    fig_single.tight_layout()
    single_save_path = os.path.join(output_dir, f"{name.lower()}_actual_vs_predicted.png")
    fig_single.savefig(single_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig_single)

    ax = axes[i]
    ax.scatter(y, y_pred, alpha=0.5, edgecolors="w", color="#2b5c8f")
    ax.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label=f"{name}: $R^2 = {r2:.3f}$")
    ax.set_xlabel("Actual HV", fontsize=fontsize)
    ax.set_ylabel("Predicted HV", fontsize=fontsize)
    ax.tick_params(axis="both", labelsize=fontsize)
    ax.legend(loc="upper left", fontsize=fontsize, frameon=True)
    ax.grid(True, linestyle=":", alpha=0.6)

for j in range(len(models), len(axes)):
    fig.delaxes(axes[j])

grid_save_path = os.path.join(output_dir, "cv_actual_vs_predicted_all.png")
fig.tight_layout()
fig.savefig(grid_save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("R2", ascending=False).reset_index(drop=True)

print("\n[INFO] Model performance ranking (sorted by R²)")
print(df_results.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

df_results.to_csv(r"./data/e_model_comparison_results.csv", index=False)

print("\n[INFO] Statistical Significance Testing (Wilcoxon Signed-Rank Test)")

best_model_name = df_results.iloc[0]["Model"]
print(f"Comparing Best Model ({best_model_name}) vs. other models based on absolute prediction errors.")
print(f"{'Comparison':<30} | {'p-value':>12} | {'Significance (a=0.05)':>25}")
print("-" * 75)

error_best = model_errors[best_model_name]

for name in models.keys():
    if name == best_model_name:
        continue
    
    error_other = model_errors[name]
    
    try:
        stat, p_value = wilcoxon(error_best, error_other, zero_method="wilcox")
        is_significant = "Significant" if p_value < 0.05 else "Not Significant"
        print(f"{best_model_name} vs {name:<14} | {p_value:>12.4e} | {is_significant:>25}")
    except ValueError:
        print(f"{best_model_name} vs {name:<14} | {'N/A':>12} | {'Identical Errors':>25}")

print(f"\n[INFO] Model benchmarking completed successfully.")
print(f"[INFO] Performance logs saved to: ./data/e_model_comparison_results.csv")
print(f"[INFO] Summary figures exported to: {output_dir}")